# 🎧 BioListen VN — Dataset Download & Preprocessing
**Team NeuraX.ai | VAIC 2026**

Notebook này tải, giải nén và tiền xử lý **4 bộ dữ liệu âm thanh sinh thái** phục vụ dự án BioListen VN.

| # | Dataset | Nguồn | Nội dung | Cách tải |
|---|---------|-------|----------|----------|
| 1 | **FSC22** | Kaggle | Tiếng cưa xích, súng, động cơ, tiếng rừng | Thủ công hoặc Kaggle API |
| 2 | **RFCx Species** | Kaggle | Chim & ếch nhái nhiệt đới (rừng mưa) | Thủ công hoặc Kaggle API |
| 3 | **AnuraSet** | Zenodo #8056090 | Ếch nhái neotropical | Tải trực tiếp qua Zenodo |
| 4 | **InsectSet459** | Zenodo #18554693 | 459 loài côn trùng | Tải trực tiếp qua Zenodo |

### ⚡ Chiến lược tải và kiểm tra dữ liệu thông minh
```
1. Kiểm tra SSD cục bộ Colab (/content/downloads/) -> Nếu có file zip, bỏ qua bước tải.
2. Kiểm tra Google Drive (BioListenVN/raw_zips/) -> Nếu có file zip, copy sang Colab bằng tqdm.
3. Nếu không có ở cả hai nơi -> Tải trực tiếp từ Internet (Zenodo/Kaggle API) kèm tqdm, sau đó backup lên Drive.
4. Giải nén trên SSD Colab cục bộ (/content/extracted/) để tối ưu hóa hiệu năng, tránh lag Drive FUSE.
5. Tiền xử lý thành spectrogram (224x224) -> lưu file HDF5 (.h5) duy nhất rồi đồng bộ ngược lên Drive.
```
> ⚠️ **QUAN TRỌNG**: KHÔNG bao giờ giải nén thẳng vào Google Drive. Hàng vạn file nhỏ sẽ treo hệ thống FUSE.

---
## 📦 Phần 1 — Thiết lập môi trường

In [ ]:
# ===========================================================
# CELL 1: Mount Google Drive & Tạo cấu trúc thư mục
# ===========================================================
from google.colab import drive
import os

print("🔗 Đang mount Google Drive...")
drive.mount('/content/drive')

# ── Thư mục LÂU DÀI trên Google Drive ──────────────────────
DRIVE_BASE      = '/content/drive/MyDrive/BioListenVN'
DRIVE_RAW       = f'{DRIVE_BASE}/raw_zips'    # Lưu file .zip gốc
DRIVE_PROCESSED = f'{DRIVE_BASE}/processed'   # Lưu file .h5
DRIVE_META      = f'{DRIVE_BASE}/metadata'    # Lưu CSV, JSON nhãn

# ── Thư mục TỐC ĐỘ CAO trên SSD cục bộ Colab ───────────────
LOCAL_DL      = '/content/downloads'
LOCAL_EXTRACT = '/content/extracted'
LOCAL_OUTPUT  = '/content/output'

for d in [DRIVE_BASE, DRIVE_RAW, DRIVE_PROCESSED, DRIVE_META,
          LOCAL_DL, LOCAL_EXTRACT, LOCAL_OUTPUT]:
    os.makedirs(d, exist_ok=True)

print('\n✅ Cấu trúc thư mục đã sẵn sàng!')
print(f'   📂 Drive backup  : {DRIVE_BASE}')
print(f'   📂 SSD Downloads : {LOCAL_DL}')
print(f'   📂 SSD Extracted : {LOCAL_EXTRACT}')
print(f'   📂 SSD Output    : {LOCAL_OUTPUT}')

In [ ]:
# ===========================================================
# CELL 2: Cài đặt thư viện cần thiết
# ===========================================================
!pip install -q librosa soundfile h5py tqdm Pillow requests kaggle

import librosa
import numpy as np
import h5py
import json
import shutil
import subprocess
import gc
from pathlib import Path
from tqdm import tqdm
from PIL import Image

print(f'✅ librosa  {librosa.__version__}')
print(f'✅ numpy    {np.__version__}')
print(f'✅ h5py     {h5py.__version__}')
print('✅ Tất cả thư viện đã sẵn sàng!')

In [ ]:
# ===========================================================
# CELL 3: Hàm tiện ích dùng chung (Sử dụng tqdm & Google Drive API)
# ===========================================================
import requests
import os
import shutil
import subprocess
import io
import json
import zipfile
from pathlib import Path
from tqdm.notebook import tqdm

# Các thư viện phục vụ Google Drive API trực tiếp
try:
    import google.auth
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
except ImportError:
    pass

_drive_service = None
_token = None

def get_drive_service():
    """ Khởi tạo và xác thực Google Drive API """
    global _drive_service, _token
    if _drive_service is None:
        print("🔗 Đang xác thực Google Drive API (Vui lòng chọn tài khoản và cho phép)...")
        auth.authenticate_user()
        creds, _ = google.auth.default()
        from google.auth.transport.requests import Request
        if not creds.valid:
            creds.refresh(Request())
        _token = creds.token
        _drive_service = build('drive', 'v3', credentials=creds)
    return _drive_service, _token


def get_relative_drive_path(absolute_path: str) -> str:
    """ Chuyển đường dẫn tuyệt đối Colab sang tương đối Drive """
    rel = absolute_path.replace('/content/drive/MyDrive', '').strip('/')
    rel = rel.replace('/content/drive/My Drive', '').strip('/')
    return rel


def get_drive_folder_id(folder_path: str) -> str:
    """ Tìm folder ID của thư mục trên Drive, tự động tạo nếu chưa có """
    drive_service, token = get_drive_service()
    rel_path = get_relative_drive_path(folder_path)
    parts = [p for p in rel_path.split('/') if p]
    parent_id = 'root'
    
    headers = {"Authorization": f"Bearer {token}"}
    
    for part in parts:
        query = f"name = '{part}' and mimeType = 'application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed = false"
        url = f"https://www.googleapis.com/drive/v3/files?q={requests.utils.quote(query)}&fields=files(id)"
        res = requests.get(url, headers=headers)
        res.raise_for_status()
        files = res.json().get('files', [])
        
        if not files:
            # Tạo thư mục mới nếu chưa có
            create_metadata = {
                "name": part,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id]
            }
            create_url = "https://www.googleapis.com/drive/v3/files"
            create_res = requests.post(
                create_url, 
                headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"}, 
                data=json.dumps(create_metadata)
            )
            create_res.raise_for_status()
            parent_id = create_res.json().get('id')
        else:
            parent_id = files[0]['id']
            
    return parent_id


def find_drive_file_id(filename: str, folder_path: str) -> str or None:
    """ Tìm ID của file trong một thư mục cụ thể trên Drive """
    try:
        drive_service, token = get_drive_service()
        folder_id = get_drive_folder_id(folder_path)
        
        query = f"name = '{filename}' and '{folder_id}' in parents and trashed = false"
        url = f"https://www.googleapis.com/drive/v3/files?q={requests.utils.quote(query)}&fields=files(id)"
        headers = {"Authorization": f"Bearer {token}"}
        res = requests.get(url, headers=headers)
        res.raise_for_status()
        files = res.json().get('files', [])
        if files:
            return files[0]['id']
    except Exception as e:
        print(f"⚠️ Không tìm thấy file {filename} qua API: {e}")
    return None


def stream_url_to_drive_api(url: str, filename: str, drive_folder_path: str) -> str:
    """
    Tải file từ URL trực tiếp lên Google Drive bằng Drive API (Resumable Upload)
    để không bị tốn dung lượng cache ổ đĩa của Colab.
    """
    drive_service, token = get_drive_service()
    folder_id = get_drive_folder_id(drive_folder_path)
    
    # Tạo phiên upload resumable
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=UTF-8",
        "X-Upload-Content-Type": "application/octet-stream",
    }
    metadata = {
        "name": filename,
        "parents": [folder_id]
    }
    init_url = "https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable"
    response = requests.post(init_url, headers=headers, data=json.dumps(metadata))
    response.raise_for_status()
    session_uri = response.headers.get("Location")
    
    # Stream từ URL
    res = requests.get(url, stream=True)
    res.raise_for_status()
    total_size = int(res.headers.get('content-length', 0))
    
    chunk_size = 10 * 1024 * 1024  # 10MB chunks
    uploaded_bytes = 0
    
    print(f"🎬 Bắt đầu stream từ URL → Drive: {filename} ({total_size / 1e9:.2f} GB)")
    
    with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Uploading {filename}") as pbar:
        iterator = res.iter_content(chunk_size=chunk_size)
        for chunk in iterator:
            if not chunk:
                break
            
            chunk_len = len(chunk)
            content_range = f"bytes {uploaded_bytes}-{uploaded_bytes + chunk_len - 1}/{total_size}"
            
            put_headers = {
                "Content-Length": str(chunk_len),
                "Content-Range": content_range,
            }
            
            put_response = requests.put(session_uri, headers=put_headers, data=chunk)
            
            if put_response.status_code in (200, 201):
                pbar.update(chunk_len)
                print(f"   ✅ Đã tải lên Drive hoàn tất: {filename}")
                return put_response.json().get('id')
            elif put_response.status_code == 308:
                uploaded_bytes += chunk_len
                pbar.update(chunk_len)
            else:
                print(f"\n❌ Lỗi upload ({put_response.status_code}): {put_response.text}")
                put_response.raise_for_status()
    return None


def download_from_drive_api(file_id: str, dest_path: str):
    """
    Tải file từ Google Drive về local SSD bằng Drive API để tránh FUSE cache.
    """
    drive_service, token = get_drive_service()
    request = drive_service.files().get_media(fileId=file_id)
    
    file_info = drive_service.files().get(fileId=file_id, fields='size').execute()
    total_size = int(file_info.get('size', 0))
    
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    fh = io.FileIO(dest_path, 'wb')
    downloader = MediaIoBaseDownload(fh, request, chunksize=10*1024*1024)
    done = False
    
    with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Downloading to local SSD") as pbar:
        last_progress = 0
        while not done:
            status, done = downloader.next_chunk()
            if status:
                progress = status.resumable_progress
                pbar.update(progress - last_progress)
                last_progress = progress
        pbar.update(total_size - last_progress)
    print(f"   ✅ Đã tải về cục bộ thành công: {dest_path}")


def download_file_tqdm(url: str, dest_path: str, desc: str = '') -> bool:
    """ Tải file bằng requests và hiển thị progress bar qua tqdm """
    label = desc or Path(dest_path).name
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024  # 1MB
        
        with open(dest_path, 'wb') as f, tqdm(
            desc=f'📥 Tải {label}',
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for data in response.iter_content(block_size):
                size = f.write(data)
                bar.update(size)
        return True
    except Exception as e:
        print(f'❌ Lỗi khi tải {label}: {e}')
        if os.path.exists(dest_path):
            os.remove(dest_path)
        return False


def copy_file_tqdm(src: str, dst: str, desc: str = '') -> None:
    """ Sao chép file lớn kèm theo thanh tiến trình tqdm """
    name = Path(src).name
    label = desc or f'Copy {name}'
    if not os.path.exists(src):
        print(f'⚠️  File nguồn không tồn tại: {src}')
        return
    
    total_size = os.path.getsize(src)
    block_size = 1024 * 1024  # 1MB
    
    try:
        with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst, tqdm(
            desc=f'💾 {label}',
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            while True:
                buf = fsrc.read(block_size)
                if not buf:
                    break
                fdst.write(buf)
                bar.update(len(buf))
        print(f'   ✅ {label} hoàn tất!')
    except Exception as e:
        print(f'❌ Lỗi copy {label}: {e}')


def safe_unzip(zip_path: str, dest_dir: str) -> bool:
    """ Giải nén file zip vào thư mục cục bộ, bỏ qua nếu đã giải nén """
    dest = Path(dest_dir)
    if dest.exists() and any(dest.iterdir()):
        print(f'   ⏭️  Đã giải nén trước đó: {dest_dir}')
        return True
    dest.mkdir(parents=True, exist_ok=True)
    name = Path(zip_path).name
    print(f'   📂 Đang giải nén {name} → {dest_dir} ...')
    r = subprocess.run(['unzip', '-q', '-o', zip_path, '-d', dest_dir],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'   ✅ Giải nén thành công: {dest_dir}')
        return True
    else:
        print(f'   ❌ Lỗi giải nén: {r.stderr[:200]}')
        return False


def find_audio_files(directory: str, exts=('.wav', '.flac', '.ogg', '.mp3')) -> list:
    """ Tìm đệ quy tất cả file âm thanh trong thư mục """
    files = []
    for root, _, fnames in os.walk(directory):
        if '__MACOSX' in root:
            continue
        for f in fnames:
            if f.lower().endswith(exts):
                files.append(os.path.join(root, f))
    return sorted(files)


print('✅ Hàm tiện ích chung & Google Drive API đã sẵn sàng!')


---
## ⬇️ Phần 2 — Tải Dataset

In [ ]:
# ===========================================================
# CELL 4: Tải Dataset 1 — FSC22 (Forest Sound Classification)
# ===========================================================
USE_KAGGLE_API = False   # ← Đổi thành True nếu muốn dùng Kaggle API

FSC22_ZIP       = f'{LOCAL_DL}/fsc22-v1.zip'
FSC22_ZIP_DRIVE = f'{DRIVE_RAW}/fsc22-v1.zip'
FSC22_DIR       = f'{LOCAL_EXTRACT}/fsc22'

print('=' * 60)
print('📥 DATASET 1: FSC22 (Forest Sound Classification)')
print('=' * 60)

if os.path.exists(FSC22_ZIP):
    print('⏭️  Đã có file zip FSC22 trên SSD Colab. Skip tải.')

elif os.path.exists(FSC22_ZIP_DRIVE):
    print('📂 Phát hiện file zip FSC22 trên Google Drive!')
    copy_file_tqdm(FSC22_ZIP_DRIVE, FSC22_ZIP, "Copy FSC22 từ Drive → SSD Colab")
    print('   ✅ Copy hoàn tất!')

elif USE_KAGGLE_API:
    # Thiết lập Kaggle API
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_json_drive = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(kaggle_json_drive):
        shutil.copy2(kaggle_json_drive, f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
        print('   ✅ Đã cấu hình Kaggle API từ Google Drive!')
        
        print('⏳ Đang tải FSC22 từ Kaggle (~2.5 GB) ...')
        os.makedirs(LOCAL_DL, exist_ok=True)
        r = subprocess.run(
            f'kaggle datasets download -d dakshinaranmal/fsc22-v1 -p {LOCAL_DL}',
            shell=True, capture_output=True, text=True
        )
        if r.returncode == 0:
            print('   ✅ Tải FSC22 thành công!')
            copy_file_tqdm(FSC22_ZIP, FSC22_ZIP_DRIVE, "Backup FSC22 lên Google Drive")
        else:
            print(f'   ❌ Lỗi Kaggle API khi tải: {r.stderr}')
    else:
        print('⚠️  Không tìm thấy kaggle.json trên Drive để dùng API!')
else:
    print('⚠️  Chưa tải được FSC22!')
    print('   💡 Cách tải: Tải thủ công fsc22-v1.zip từ Kaggle và lưu vào Google Drive tại:')
    print(f'      {FSC22_ZIP_DRIVE}')

# Giải nén cục bộ
if os.path.exists(FSC22_ZIP):
    safe_unzip(FSC22_ZIP, FSC22_DIR)
    print('\n📊 Thống kê FSC22:')
    !find "{FSC22_DIR}" -type f \( -name "*.wav" -o -name "*.flac" \) 2>/dev/null | wc -l | xargs echo "   File âm thanh:"
else:
    print('\n⏭️  Bỏ qua giải nén FSC22 — chưa có file zip.')

In [ ]:
# ===========================================================
# CELL 5: Tải Dataset 2 — RFCx Species Audio Detection
# ===========================================================
USE_KAGGLE_API = False   # ← Đổi thành True nếu muốn dùng Kaggle API

RFCX_ZIP      = f'{LOCAL_DL}/rfcx-species-audio-detection.zip'
RFCX_ZIP_DRIVE = f'{DRIVE_RAW}/rfcx-species-audio-detection.zip'
RFCX_DIR      = f'{LOCAL_EXTRACT}/rfcx'

print('=' * 60)
print('📥 DATASET 2: RFCx Species Audio Detection')
print('=' * 60)

if os.path.exists(RFCX_ZIP):
    print('⏭️  Đã có file zip RFCx trên SSD Colab. Skip tải.')

elif os.path.exists(RFCX_ZIP_DRIVE):
    print('📂 Phát hiện file zip RFCx trên Google Drive!')
    copy_file_tqdm(RFCX_ZIP_DRIVE, RFCX_ZIP, "Copy RFCx từ Drive → SSD Colab")
    print('   ✅ Copy hoàn tất!')

elif USE_KAGGLE_API:
    # Thiết lập Kaggle API
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_json_drive = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(kaggle_json_drive):
        shutil.copy2(kaggle_json_drive, f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
        
        print('⏳ Đang tải RFCx bằng Kaggle API (~7.5 GB) ...')
        os.makedirs(LOCAL_DL, exist_ok=True)
        r = subprocess.run(
            f'kaggle competitions download -c rfcx-species-audio-detection -p {LOCAL_DL}',
            shell=True, capture_output=True, text=True
        )
        if r.returncode == 0:
            print('   ✅ Tải RFCx thành công!')
            copy_file_tqdm(RFCX_ZIP, RFCX_ZIP_DRIVE, "Backup RFCx lên Google Drive")
        else:
            print(f'   ❌ Lỗi Kaggle API khi tải: {r.stderr}')
            print('   💡 Thử Accept Rules tại: https://www.kaggle.com/c/rfcx-species-audio-detection/rules')
    else:
        print('⚠️  Không tìm thấy kaggle.json trên Drive để dùng API!')
else:
    print('⚠️  Chưa tải được RFCx!')
    print('   💡 Cách tải: Tải thủ công rfcx-species-audio-detection.zip từ Kaggle và lưu vào Google Drive tại:')
    print(f'      {RFCX_ZIP_DRIVE}')

# Giải nén cục bộ
if os.path.exists(RFCX_ZIP):
    safe_unzip(RFCX_ZIP, RFCX_DIR)
    print('\n📊 Thống kê RFCx:')
    !find "{RFCX_DIR}" -type f \( -name "*.flac" -o -name "*.wav" \) 2>/dev/null | wc -l | xargs echo "   File âm thanh:"
    !find "{RFCX_DIR}" -name "*.csv" 2>/dev/null | head -5 | xargs echo "   CSV files:"
else:
    print('\n⏭️  Bỏ qua giải nén RFCx — chưa có file zip.')

In [ ]:
# ===========================================================
# CELL 6: Tải Dataset 3 — AnuraSet (Ếch nhái)
# ===========================================================
ANURA_ZIP       = f'{LOCAL_DL}/anuraset.zip'
ANURA_ZIP_DRIVE = f'{DRIVE_RAW}/anuraset.zip'
ANURA_LABELS    = f'{LOCAL_DL}/anuraset_weak_labels.csv'
ANURA_DIR       = f'{LOCAL_EXTRACT}/anuraset'

ZENODO_ANURA = 8056090

print('=' * 60)
print('📥 DATASET 3: AnuraSet (Zenodo #8056090)')
print('=' * 60)

# ── Bước 1: Kiểm tra & Tải file zip chính từ Zenodo ─────────
if os.path.exists(ANURA_ZIP):
    print('⏭️  Đã có file zip AnuraSet trên SSD Colab. Skip tải.')

elif os.path.exists(ANURA_ZIP_DRIVE):
    print('📂 Phát hiện file zip AnuraSet trên Google Drive!')
    copy_file_tqdm(ANURA_ZIP_DRIVE, ANURA_ZIP, "Copy AnuraSet từ Drive → SSD Colab")
    print('   ✅ Copy hoàn tất!')

else:
    print('⏳ Không có sẵn file zip. Tải trực tiếp từ Zenodo...')
    download_file_tqdm(
        url=f'https://zenodo.org/records/{ZENODO_ANURA}/files/anuraset.zip?download=1',
        dest_path=ANURA_ZIP,
        desc='AnuraSet (anuraset.zip)'
    )
    copy_file_tqdm(ANURA_ZIP, ANURA_ZIP_DRIVE, "Backup AnuraSet lên Google Drive")

# ── Bước 2: Tải file nhãn CSV ──────────────────────────────
if not os.path.exists(ANURA_LABELS):
    kaggle_csv_drive = f'{DRIVE_META}/anuraset_weak_labels.csv'
    if os.path.exists(kaggle_csv_drive):
        print('📂 Phát hiện CSV weak labels trên Drive. Đang copy sang Colab...')
        shutil.copy2(kaggle_csv_drive, ANURA_LABELS)
    else:
        print('⏳ Tải file nhãn weak_labels.csv từ Zenodo...')
        download_file_tqdm(
            url=f'https://zenodo.org/records/{ZENODO_ANURA}/files/weak_labels.csv?download=1',
            dest_path=ANURA_LABELS,
            desc='AnuraSet Labels (weak_labels.csv)'
        )
        copy_file_tqdm(ANURA_LABELS, kaggle_csv_drive, "Backup weak_labels.csv lên Drive")
else:
    print('⏭️  Đã có: weak_labels.csv')

# ── Bước 3: Giải nén cục bộ ────────────────────────────────
if os.path.exists(ANURA_ZIP):
    safe_unzip(ANURA_ZIP, ANURA_DIR)
    print('\n📊 Thống kê AnuraSet:')
    !find "{ANURA_DIR}" -maxdepth 2 -type d 2>/dev/null | head -10
    !find "{ANURA_DIR}" -type f \( -name "*.wav" -o -name "*.flac" \) 2>/dev/null | wc -l | xargs echo "   Tổng file âm thanh:"
else:
    print('\n⏭️  Bỏ qua giải nén AnuraSet — chưa có file zip.')

In [ ]:
# ===========================================================
# CELL 7: Tải Dataset 4 — InsectSet459 (Côn trùng)
# ===========================================================
INSECT_DL_DIR          = f'{LOCAL_DL}/insectset459'
INSECT_CSV             = f'{INSECT_DL_DIR}/Annotation.csv'

INSECT_TRAIN_ZIP_DRIVE = f'{DRIVE_RAW}/insect_Train.zip'
INSECT_VAL_ZIP_DRIVE   = f'{DRIVE_RAW}/insect_Validation.zip'
INSECT_CSV_DRIVE       = f'{DRIVE_META}/insect_Annotation.csv'

INSECT_DIR             = f'{LOCAL_EXTRACT}/insectset459'

ZENODO_INSECT = 18554693
os.makedirs(INSECT_DL_DIR, exist_ok=True)

print('=' * 60)
print('📥 DATASET 4: InsectSet459 (Zenodo #18554693)')
print('=' * 60)

# Khởi tạo API trước
get_drive_service()

# ── Bước 1: Tải Train.zip trực tiếp về Google Drive bằng API ────────
train_id = find_drive_file_id('insect_Train.zip', DRIVE_RAW)
if os.path.exists(INSECT_TRAIN_ZIP_DRIVE) or train_id:
    print('📂 Đã có insect_Train.zip trên Google Drive. Skip tải.')
else:
    print('⏳ Tải Train.zip trực tiếp về Google Drive từ Zenodo bằng API...')
    stream_url_to_drive_api(
        url=f'https://zenodo.org/records/{ZENODO_INSECT}/files/Train.zip?download=1',
        filename='insect_Train.zip',
        drive_folder_path=DRIVE_RAW
    )

# ── Bước 2: Tải Validation.zip trực tiếp về Google Drive bằng API ───
val_id = find_drive_file_id('insect_Validation.zip', DRIVE_RAW)
if os.path.exists(INSECT_VAL_ZIP_DRIVE) or val_id:
    print('📂 Đã có insect_Validation.zip trên Google Drive. Skip tải.')
else:
    print('⏳ Tải Validation.zip trực tiếp về Google Drive từ Zenodo bằng API...')
    stream_url_to_drive_api(
        url=f'https://zenodo.org/records/{ZENODO_INSECT}/files/Validation.zip?download=1',
        filename='insect_Validation.zip',
        drive_folder_path=DRIVE_RAW
    )

# ── Bước 3: Tải Annotation CSV ─────────────────────────────
csv_id = find_drive_file_id('insect_Annotation.csv', DRIVE_META)
if os.path.exists(INSECT_CSV):
    print('⏭️  Đã có Annotation.csv trên SSD Colab. Skip.')
elif os.path.exists(INSECT_CSV_DRIVE) or csv_id:
    print('📂 Phát hiện Annotation.csv trên Google Drive!')
    if not os.path.exists(INSECT_CSV_DRIVE) and csv_id:
        download_from_drive_api(csv_id, INSECT_CSV)
    else:
        shutil.copy2(INSECT_CSV_DRIVE, INSECT_CSV)
else:
    print('⏳ Tải Annotation.csv trực tiếp về Google Drive bằng API...')
    stream_url_to_drive_api(
        url=f'https://zenodo.org/records/{ZENODO_INSECT}/files/InsectSet459_Train_Val_Test_Annotation.csv?download=1',
        filename='insect_Annotation.csv',
        drive_folder_path=DRIVE_META
    )
    shutil.copy2(INSECT_CSV_DRIVE, INSECT_CSV)

print('\nℹ️ Giải pháp On-Demand ZIP Processing đã sẵn sàng. Các file ZIP lớn sẽ được tải cục bộ từng phần và tiền xử lý tại CELL 13.')


In [ ]:
# ===========================================================
# CELL 8: Tổng kết dung lượng đã tải
# ===========================================================
print('=' * 60)
print('📊 TỔNG KẾT DUNG LƯỢNG ĐÃ TẢI')
print('=' * 60)

print('\n--- SSD Colab (Downloads) ---')
!du -sh /content/downloads/* 2>/dev/null || echo '  (trống)'

print('\n--- SSD Colab (Extracted) ---')
!du -sh /content/extracted/* 2>/dev/null || echo '  (trống)'

print('\n--- Google Drive (raw_zips backup) ---')
!du -sh /content/drive/MyDrive/BioListenVN/raw_zips/* 2>/dev/null || echo '  (trống)'

print('\n--- SSD còn lại ---')
!df -h /content | tail -1

---
## 🔬 Phần 3 — Tiền xử lý: Audio → Mel-Spectrogram → HDF5

Pipeline xử lý mỗi file âm thanh:
```
Audio (.wav/.flac)
  → Resample 22,050 Hz
  → Pad / Trim → 5 giây cố định (110,250 samples)
  → Mel-Spectrogram (128 mel bins, n_fft=2048, hop=512)
  → Chuyển sang dB scale → Normalize [0, 255] uint8
  → Resize (224 × 224)
  → Lưu vào HDF5
```

| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| `sample_rate` | 22,050 Hz | Đủ cho dải tần sinh thái (< 11 kHz) |
| `duration` | 5 s | Chuẩn hoá độ dài đầu vào |
| `n_fft` | 2,048 | Cân bằng độ phân giải tần số/thời gian |
| `hop_length` | 512 | Overlap ~75% |
| `n_mels` | 128 | Đủ chi tiết cho CNN |
| Output | 224 × 224 uint8 | Tương thích EfficientNet-V2-S |

In [ ]:
# ===========================================================

def load_audio_from_zip_obj(z_obj, filepath_in_zip: str) -> np.ndarray | None:
    """
    Tải file âm thanh trực tiếp từ một đối tượng ZipFile đang mở,
    resample về SR Hz, pad/trim về N_SAMP samples.
    """
    try:
        with z_obj.open(filepath_in_zip) as f:
            audio_data = io.BytesIO(f.read())
            y, _ = librosa.load(audio_data, sr=SR, mono=True)
            if len(y) < N_SAMP:
                y = np.pad(y, (0, N_SAMP - len(y)), mode='constant')
            else:
                y = y[:N_SAMP]
            return y.astype(np.float32)
    except Exception:
        return None


def process_zip_dataset(zip_path: str, audio_files_in_zip: list, labels: list,
                        output_h5: str, dataset_name: str,
                        class_map: dict = None,
                        batch_size: int = 512) -> str | None:
    """
    Xử lý danh sách file âm thanh trực tiếp từ file ZIP -> HDF5.
    """
    specs_buf, labels_buf = [], []
    n_ok, n_err = 0, 0
    
    # Xác định HDF5 file mode: ghi mới hay append
    mode = 'a' if os.path.exists(output_h5) else 'w'
    first_write = (mode == 'w')

    def _flush_to_hdf5():
        nonlocal first_write
        if not specs_buf:
            return
        specs_arr  = np.stack(specs_buf)
        labels_arr = np.array(labels_buf, dtype=np.int32)

        with h5py.File(output_h5, 'w' if first_write else 'a') as hf:
            if first_write:
                hf.create_dataset(
                    'spectrograms',
                    data=specs_arr,
                    compression='gzip', compression_opts=4,
                    chunks=(min(64, len(specs_arr)), IMG_SIZE, IMG_SIZE),
                    maxshape=(None, IMG_SIZE, IMG_SIZE)
                )
                hf.create_dataset(
                    'labels', data=labels_arr,
                    maxshape=(None,)
                )
                hf.attrs['dataset_name'] = dataset_name
                hf.attrs['sample_rate']  = SR
                hf.attrs['duration_sec'] = DURATION
                hf.attrs['n_mels']       = N_MELS
                hf.attrs['n_fft']        = N_FFT
                hf.attrs['hop_length']   = HOP
                hf.attrs['img_size']     = IMG_SIZE
                if class_map:
                    hf.attrs['class_map'] = json.dumps(
                        {str(k): v for k, v in class_map.items()}
                    )
                first_write = False
            else:
                old_n = hf['spectrograms'].shape[0]
                new_n = old_n + len(specs_arr)
                hf['spectrograms'].resize(new_n, axis=0)
                hf['labels'].resize(new_n, axis=0)
                hf['spectrograms'][old_n:] = specs_arr
                hf['labels'][old_n:]       = labels_arr

        specs_buf.clear()
        labels_buf.clear()
        gc.collect()

    with zipfile.ZipFile(zip_path, 'r') as z_obj:
        pbar = tqdm(zip(audio_files_in_zip, labels),
                    total=len(audio_files_in_zip), desc=f'🔄 {dataset_name} (ZIP)')
        for fpath, label in pbar:
            y = load_audio_from_zip_obj(z_obj, fpath)
            if y is None:
                n_err += 1
                continue
            spec = waveform_to_spectrogram(y)
            specs_buf.append(spec)
            labels_buf.append(label)
            n_ok += 1
            del y, spec

            if len(specs_buf) >= batch_size:
                _flush_to_hdf5()
                pbar.set_postfix({'ok': n_ok, 'err': n_err})

        _flush_to_hdf5()

    if n_ok == 0:
        print(f'\n❌ {dataset_name}: Không xử lý được mẫu nào!')
        return None

    size_mb = os.path.getsize(output_h5) / 1e6
    print(f'\n✅ {dataset_name}: {n_ok:,} mẫu → {output_h5}')
    print(f'   Size: {size_mb:.1f} MB | Bỏ qua: {n_err}')
    return output_h5

# CELL 9: Pipeline tiền xử lý cốt lõi
# ===========================================================

# ── Cấu hình (khớp với TASKS_VIET.md) ──────────────────────
SR       = 22_050   # Hz
DURATION = 5        # giây
N_SAMP   = SR * DURATION  # = 110,250 samples
N_FFT    = 2048
HOP      = 512
N_MELS   = 128
IMG_SIZE = 224      # Chiều cao và chiều rộng spectrogram đầu ra


def load_and_pad_audio(filepath: str) -> np.ndarray | None:
    """
    Tải file âm thanh, resample về SR Hz, pad/trim về N_SAMP samples.
    Trả về numpy array (N_SAMP,) hoặc None nếu lỗi.
    """
    try:
        y, _ = librosa.load(filepath, sr=SR, mono=True)
        if len(y) < N_SAMP:
            y = np.pad(y, (0, N_SAMP - len(y)), mode='constant')
        else:
            y = y[:N_SAMP]
        return y.astype(np.float32)
    except Exception:
        return None


def load_segment(filepath: str, offset: float) -> np.ndarray | None:
    """
    Tải đoạn DURATION giây từ vị trí offset trong file dài (dùng cho RFCx).
    """
    try:
        y, _ = librosa.load(filepath, sr=SR, mono=True,
                            offset=offset, duration=DURATION)
        if len(y) < N_SAMP:
            y = np.pad(y, (0, N_SAMP - len(y)), mode='constant')
        else:
            y = y[:N_SAMP]
        return y.astype(np.float32)
    except Exception:
        return None


def waveform_to_spectrogram(y: np.ndarray) -> np.ndarray:
    """
    Chuyển waveform → Mel-Spectrogram dB → Resize (IMG_SIZE × IMG_SIZE) uint8.
    Output shape: (IMG_SIZE, IMG_SIZE) — ảnh grayscale 1 channel
    """
    # 1. Tính Mel-Spectrogram
    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS
    )
    # 2. Chuyển sang dB scale (logarithmic)
    S_db = librosa.power_to_db(S, ref=np.max)
    # 3. Normalize về [0, 1]
    s_min, s_max = S_db.min(), S_db.max()
    S_norm = (S_db - s_min) / (s_max - s_min + 1e-8)
    # 4. Resize về (IMG_SIZE, IMG_SIZE) bằng Pillow
    img = Image.fromarray((S_norm * 255).astype(np.uint8))
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.array(img, dtype=np.uint8)


def process_dataset(audio_files: list, labels: list,
                    output_h5: str, dataset_name: str,
                    class_map: dict = None,
                    offsets: list = None,
                    batch_size: int = 512) -> str | None:
    """
    Xử lý danh sách file âm thanh → HDF5.
    """
    if offsets is None:
        offsets = [None] * len(audio_files)

    specs_buf, labels_buf = [], []
    n_ok, n_err = 0, 0
    first_write = True

    def _flush_to_hdf5():
        nonlocal first_write
        if not specs_buf:
            return
        specs_arr  = np.stack(specs_buf)
        labels_arr = np.array(labels_buf, dtype=np.int32)

        mode = 'w' if first_write else 'a'
        with h5py.File(output_h5, mode) as hf:
            if first_write:
                hf.create_dataset(
                    'spectrograms',
                    data=specs_arr,
                    compression='gzip', compression_opts=4,
                    chunks=(min(64, len(specs_arr)), IMG_SIZE, IMG_SIZE),
                    maxshape=(None, IMG_SIZE, IMG_SIZE)
                )
                hf.create_dataset(
                    'labels', data=labels_arr,
                    maxshape=(None,)
                )
                hf.attrs['dataset_name'] = dataset_name
                hf.attrs['sample_rate']  = SR
                hf.attrs['duration_sec'] = DURATION
                hf.attrs['n_mels']       = N_MELS
                hf.attrs['n_fft']        = N_FFT
                hf.attrs['hop_length']   = HOP
                hf.attrs['img_size']     = IMG_SIZE
                if class_map:
                    hf.attrs['class_map'] = json.dumps(
                        {str(k): v for k, v in class_map.items()}
                    )
                first_write = False
            else:
                old_n = hf['spectrograms'].shape[0]
                new_n = old_n + len(specs_arr)
                hf['spectrograms'].resize(new_n, axis=0)
                hf['labels'].resize(new_n, axis=0)
                hf['spectrograms'][old_n:] = specs_arr
                hf['labels'][old_n:]       = labels_arr

        specs_buf.clear()
        labels_buf.clear()
        gc.collect()

    pbar = tqdm(zip(audio_files, labels, offsets),
                total=len(audio_files), desc=f'🔄 {dataset_name}')
    for fpath, label, offset in pbar:
        y = (load_segment(fpath, offset) if offset is not None
             else load_and_pad_audio(fpath))
        if y is None:
            n_err += 1
            continue
        spec = waveform_to_spectrogram(y)
        specs_buf.append(spec)
        labels_buf.append(label)
        n_ok += 1
        del y, spec

        if len(specs_buf) >= batch_size:
            _flush_to_hdf5()
            pbar.set_postfix({'ok': n_ok, 'err': n_err})

    _flush_to_hdf5()

    if n_ok == 0:
        print(f'\n❌ {dataset_name}: Không xử lý được mẫu nào!')
        return None

    size_mb = os.path.getsize(output_h5) / 1e6
    print(f'\n✅ {dataset_name}: {n_ok:,} mẫu → {output_h5}')
    print(f'   Size: {size_mb:.1f} MB | Bỏ qua: {n_err}')
    return output_h5


print('✅ Pipeline tiền xử lý đã sẵn sàng!')

In [ ]:
# ===========================================================
# CELL 10: Tiền xử lý FSC22 → fsc22_melspec.h5
# ===========================================================
FSC22_DIR = f'{LOCAL_EXTRACT}/fsc22'
FSC22_H5 = f'{LOCAL_OUTPUT}/fsc22_melspec.h5'

if not os.path.exists(FSC22_DIR) or not any(Path(FSC22_DIR).iterdir()):
    print('⏸️  Bỏ qua FSC22 — chưa có file dữ liệu.')
else:
    print('🔍 Đang quét cấu trúc FSC22 ...')
    fsc22_data_dir = FSC22_DIR
    subdirs = [d for d in os.listdir(fsc22_data_dir) if os.path.isdir(os.path.join(fsc22_data_dir, d))]
    if len(subdirs) == 1:
        fsc22_data_dir = os.path.join(fsc22_data_dir, subdirs[0])
        
    fsc22_files = find_audio_files(fsc22_data_dir)
    print(f'   Tìm thấy {len(fsc22_files)} file âm thanh.')
    
    # Nhãn được xác định từ tên thư mục cha
    fsc22_class_names = sorted(list(set(os.path.basename(os.path.dirname(f)) for f in fsc22_files)))
    fsc22_class_to_idx = {name: idx for idx, name in enumerate(fsc22_class_names)}
    fsc22_labels = [fsc22_class_to_idx[os.path.basename(os.path.dirname(f))] for f in fsc22_files]
    fsc22_idx_to_class = {v: k for k, v in fsc22_class_to_idx.items()}
    
    print(f'   Danh sách classes ({len(fsc22_class_names)}): {fsc22_class_names}')
    
    # Lưu mappings
    with open(f'{LOCAL_OUTPUT}/fsc22_classes.json', 'w') as f:
        json.dump({'class_map': fsc22_idx_to_class, 'class_to_idx': fsc22_class_to_idx}, f, indent=2)
        
    # Chạy tiền xử lý
    process_dataset(
        audio_files = fsc22_files,
        labels = fsc22_labels,
        output_h5 = FSC22_H5,
        dataset_name = 'FSC22',
        class_map = fsc22_idx_to_class
    )
    del fsc22_files, fsc22_labels
    gc.collect()

In [ ]:
# ===========================================================
# CELL 11: Tiền xử lý RFCx → rfcx_melspec.h5
# ===========================================================
import pandas as pd
RFCX_DIR = f'{LOCAL_EXTRACT}/rfcx'
RFCX_H5 = f'{LOCAL_OUTPUT}/rfcx_melspec.h5'

if not os.path.exists(RFCX_DIR) or not any(Path(RFCX_DIR).iterdir()):
    print('⏸️  Bỏ qua RFCx — chưa tải xong.')
else:
    tp_csv = None
    for root, _, fnames in os.walk(RFCX_DIR):
        if 'train_tp.csv' in fnames:
            tp_csv = os.path.join(root, 'train_tp.csv')
            break

    if tp_csv is None:
        print('❌ Không tìm thấy train_tp.csv trong RFCx!')
    else:
        df = pd.read_csv(tp_csv)
        rfcx_audio_dir = None
        for candidate in [f'{RFCX_DIR}/train', RFCX_DIR]:
            if find_audio_files(candidate):
                rfcx_audio_dir = candidate
                break

        if rfcx_audio_dir is None:
            print('❌ Không tìm thấy file audio trong RFCx!')
        else:
            rfcx_files, rfcx_labels, rfcx_offsets = [], [], []
            missing = 0
            for _, row in df.iterrows():
                rec_id  = row['recording_id']
                species = int(row['species_id'])
                t_min   = float(row['t_min'])
                found = None
                for ext in ('.flac', '.wav', '.ogg'):
                    p = os.path.join(rfcx_audio_dir, f'{rec_id}{ext}')
                    if os.path.exists(p):
                        found = p
                        break
                if found:
                    rfcx_files.append(found)
                    rfcx_labels.append(species)
                    rfcx_offsets.append(t_min)
                else:
                    missing += 1

            print(f'   Tìm thấy: {len(rfcx_files)} mẫu hợp lệ | Thiếu file: {missing}')

            unique_sp = sorted(set(rfcx_labels))
            sp_remap  = {old: new for new, old in enumerate(unique_sp)}
            rfcx_labels_remapped = [sp_remap[l] for l in rfcx_labels]
            rfcx_class_map = {v: f'species_{k}' for k, v in sp_remap.items()}

            rfcx_meta_json = f'{LOCAL_OUTPUT}/rfcx_classes.json'
            with open(rfcx_meta_json, 'w') as f:
                json.dump({'class_map': rfcx_class_map, 'species_remap': sp_remap}, f, indent=2)

            process_dataset(
                audio_files  = rfcx_files,
                labels       = rfcx_labels_remapped,
                output_h5    = RFCX_H5,
                dataset_name = 'RFCx',
                class_map    = rfcx_class_map,
                offsets      = rfcx_offsets,
            )
            del df, rfcx_files, rfcx_labels, rfcx_offsets
            gc.collect()

In [ ]:
# ===========================================================
# CELL 12: Tiền xử lý AnuraSet → anuraset_melspec.h5
# ===========================================================
ANURA_DIR = f'{LOCAL_EXTRACT}/anuraset'
ANURA_H5 = f'{LOCAL_OUTPUT}/anuraset_melspec.h5'

print('🔍 Đang quét cấu trúc AnuraSet ...')
anura_files = find_audio_files(ANURA_DIR)
print(f'   Tổng file âm thanh: {len(anura_files)}')

if len(anura_files) == 0:
    print('⏸️  Bỏ qua AnuraSet — không tìm thấy file âm thanh.')
else:
    parent_dirs = [os.path.basename(os.path.dirname(f)) for f in anura_files]
    unique_dirs = sorted(set(parent_dirs))

    if len(unique_dirs) > 1:
        dir_to_idx  = {d: i for i, d in enumerate(unique_dirs)}
        anura_labels = [dir_to_idx[d] for d in parent_dirs]
        anura_class_map = {v: k for k, v in dir_to_idx.items()}
        print(f'   {len(unique_dirs)} classes từ cấu trúc thư mục')
    else:
        print('   Cấu trúc flat — gán nhãn 0')
        anura_labels    = [0] * len(anura_files)
        anura_class_map = {0: 'anuran'}

    anura_meta_json = f'{LOCAL_OUTPUT}/anuraset_classes.json'
    with open(anura_meta_json, 'w') as f:
        json.dump({'class_map': {str(k): v for k, v in anura_class_map.items()}}, f, indent=2)

    process_dataset(
        audio_files  = anura_files,
        labels       = anura_labels,
        output_h5    = ANURA_H5,
        dataset_name = 'AnuraSet',
        class_map    = anura_class_map,
    )
    del anura_files, anura_labels
    gc.collect()

In [ ]:
# ===========================================================
# CELL 13: Tiền xử lý InsectSet459 → insect_melspec.h5 (ZIP On-Demand)
# ===========================================================
import gc
INSECT_H5 = f'{LOCAL_OUTPUT}/insect_melspec.h5'

print('🔍 Đang khởi tạo quét cấu trúc ZIP từ Google Drive...')
get_drive_service()

# 1. Đọc danh sách file trực tiếp từ ZIP trên Google Drive (chỉ đọc Central Directory, cực nhanh & không cache)
print('   📂 Quét file từ insect_Train.zip trên Drive...')
with zipfile.ZipFile(INSECT_TRAIN_ZIP_DRIVE, 'r') as z:
    train_names = z.namelist()

print('   📂 Quét file từ insect_Validation.zip trên Drive...')
with zipfile.ZipFile(INSECT_VAL_ZIP_DRIVE, 'r') as z:
    val_names = z.namelist()

all_names = train_names + val_names
audio_names = [f for f in all_names if f.lower().endswith(('.wav', '.flac', '.mp3', '.ogg')) and '__MACOSX' not in f]

if len(audio_names) == 0:
    print('⏭️  Bỏ qua InsectSet459 — không tìm thấy file âm thanh.')
else:
    # 2. Tạo ánh xạ nhãn class
    parent_dirs = [os.path.basename(os.path.dirname(f)) for f in audio_names]
    unique_dirs = sorted(set(parent_dirs))

    if len(unique_dirs) > 1:
        dir_to_idx    = {d: i for i, d in enumerate(unique_dirs)}
        insect_class_map = {v: k for k, v in dir_to_idx.items()}
        print(f'   Phát hiện {len(unique_dirs)} species.')
    else:
        dir_to_idx    = {}
        insect_class_map = {0: 'insect'}

    insect_meta_json = f'{LOCAL_OUTPUT}/insect_classes.json'
    with open(insect_meta_json, 'w') as f:
        json.dump({'class_map': {str(k): v for k, v in insect_class_map.items()}}, f, indent=2)

    # 3. Tiến hành tải và tiền xử lý Train.zip
    LOCAL_TRAIN_ZIP = f'{LOCAL_DL}/insect_Train.zip'
    train_id = find_drive_file_id('insect_Train.zip', DRIVE_RAW)
    print('\n📥 Tải file Train.zip về cục bộ SSD Colab...')
    if train_id:
        download_from_drive_api(train_id, LOCAL_TRAIN_ZIP)
    else:
        download_file_tqdm(
            url=f'https://zenodo.org/records/{ZENODO_INSECT}/files/Train.zip?download=1',
            dest_path=LOCAL_TRAIN_ZIP,
            desc='Train.zip'
        )

    # Lọc danh sách file train
    train_audio_files = [f for f in train_names if f.lower().endswith(('.wav', '.flac', '.mp3', '.ogg')) and '__MACOSX' not in f]
    train_labels = [dir_to_idx[os.path.basename(os.path.dirname(f))] if dir_to_idx else 0 for f in train_audio_files]

    print('\n⚙️ Đang tiền xử lý Train set...')
    # Xóa file H5 cũ nếu tồn tại trước khi ghi mới
    if os.path.exists(INSECT_H5):
        os.remove(INSECT_H5)
        
    process_zip_dataset(
        zip_path = LOCAL_TRAIN_ZIP,
        audio_files_in_zip = train_audio_files,
        labels = train_labels,
        output_h5 = INSECT_H5,
        dataset_name = 'InsectSet459',
        class_map = insect_class_map
    )

    # Giải phóng không gian SSD của Train.zip ngay lập tức!
    print('\n🗑️ Đang xóa local Train.zip để giải phóng dung lượng SSD...')
    if os.path.exists(LOCAL_TRAIN_ZIP):
        os.remove(LOCAL_TRAIN_ZIP)
    gc.collect()

    # 4. Tiến hành tải và tiền xử lý Validation.zip
    LOCAL_VAL_ZIP = f'{LOCAL_DL}/insect_Validation.zip'
    val_id = find_drive_file_id('insect_Validation.zip', DRIVE_RAW)
    print('\n📥 Tải file Validation.zip về cục bộ SSD Colab...')
    if val_id:
        download_from_drive_api(val_id, LOCAL_VAL_ZIP)
    else:
        download_file_tqdm(
            url=f'https://zenodo.org/records/{ZENODO_INSECT}/files/Validation.zip?download=1',
            dest_path=LOCAL_VAL_ZIP,
            desc='Validation.zip'
        )

    # Lọc danh sách file val
    val_audio_files = [f for f in val_names if f.lower().endswith(('.wav', '.flac', '.mp3', '.ogg')) and '__MACOSX' not in f]
    val_labels = [dir_to_idx[os.path.basename(os.path.dirname(f))] if dir_to_idx else 0 for f in val_audio_files]

    print('\n⚙️ Đang tiền xử lý Validation set...')
    process_zip_dataset(
        zip_path = LOCAL_VAL_ZIP,
        audio_files_in_zip = val_audio_files,
        labels = val_labels,
        output_h5 = INSECT_H5,
        dataset_name = 'InsectSet459',
        class_map = insect_class_map
    )

    # Giải phóng không gian SSD của Validation.zip ngay lập tức!
    print('\n🗑️ Đang xóa local Validation.zip để giải phóng dung lượng SSD...')
    if os.path.exists(LOCAL_VAL_ZIP):
        os.remove(LOCAL_VAL_ZIP)
    
    del train_names, val_names, train_audio_files, val_audio_files
    gc.collect()
    print('\n✅ Hoàn tất toàn bộ pipeline xử lý InsectSet459!')


---
## 💾 Phần 4 — Backup kết quả lên Google Drive

In [ ]:
# ===========================================================
# CELL 14: Copy tất cả file .h5 và metadata lên Drive
# ===========================================================
import glob

print('=' * 60)
print('💾 BACKUP KẾT QUẢ LÊN GOOGLE DRIVE')
print('=' * 60)

print('\n📤 Copy file HDF5 ...')
h5_files = glob.glob(f'{LOCAL_OUTPUT}/*.h5')
for src in sorted(h5_files):
    dest = f'{DRIVE_PROCESSED}/{Path(src).name}'
    copy_file_tqdm(src, dest)

print('\n📤 Copy file metadata JSON ...')
json_files = glob.glob(f'{LOCAL_OUTPUT}/*.json')
for src in sorted(json_files):
    dest = f'{DRIVE_META}/{Path(src).name}'
    copy_file_tqdm(src, dest)

print('\n✅ Backup hoàn tất!')

In [ ]:
# ===========================================================
# CELL 15: Xác minh & Tổng kết
# ===========================================================
import glob
print('=' * 60)
print('📊 TỔNG KẾT — BioListen VN Datasets')
print('=' * 60)

drive_h5 = sorted(glob.glob(f'{DRIVE_PROCESSED}/*.h5'))
total_samples = 0

if not drive_h5:
    print('\n⚠️  Chưa có file .h5 nào trên Drive. Hãy chạy các Cell tiền xử lý trước.')
else:
    print()
    for h5_path in drive_h5:
        size_mb = os.path.getsize(h5_path) / 1e6
        with h5py.File(h5_path, 'r') as hf:
            n  = hf['spectrograms'].shape[0]
            nc = len(set(hf['labels'][:]))
            sh = hf['spectrograms'].shape
            ds = hf.attrs.get('dataset_name', '?')
            total_samples += n
            print(f'  📁 {Path(h5_path).name}')
            print(f'     Dataset: {ds} | Mẫu: {n:,} | Classes: {nc}')
            print(f'     Shape: {sh} | Size: {size_mb:.1f} MB')
            print()

    print(f'  🎯 TỔNG CỘNG: {total_samples:,} mẫu Mel-Spectrogram đã xử lý thành công!')

print('\n--- Google Drive BioListenVN ---')
!ls -lh "{DRIVE_BASE}/" 2>/dev/null
print('\n--- processed/ ---')
!ls -lh "{DRIVE_PROCESSED}/" 2>/dev/null
print('\n--- raw_zips/ ---')
!ls -lh "{DRIVE_RAW}/" 2>/dev/null
print('\n--- metadata/ ---')
!ls -lh "{DRIVE_META}/" 2>/dev/null

---
## 📝 Ghi chú — Cách dùng HDF5 trong Training Notebook

```python
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class BioListenDataset(Dataset):
    """
    PyTorch Dataset đọc trực tiếp từ file HDF5.
    Chuyển ảnh grayscale (224, 224) uint8
    → 3-channel float32 tensor (3, 224, 224) cho EfficientNet.
    """
    def __init__(self, h5_path: str):
        self.hf     = h5py.File(h5_path, 'r')
        self.specs  = self.hf['spectrograms']  # Lazy loading
        self.labels = self.hf['labels'][:]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        # (224, 224) uint8 → (3, 224, 224) float32
        spec = self.specs[idx].astype('float32') / 255.0
        spec = np.stack([spec, spec, spec], axis=0)  # 3 channels
        return torch.tensor(spec), torch.tensor(int(self.labels[idx]))

    def close(self):
        self.hf.close()

# Sử dụng
DRIVE_PROCESSED = '/content/drive/MyDrive/BioListenVN/processed'

ds_fsc22 = BioListenDataset(f'{DRIVE_PROCESSED}/fsc22_melspec.h5')
ds_rfcx  = BioListenDataset(f'{DRIVE_PROCESSED}/rfcx_melspec.h5')
ds_anura = BioListenDataset(f'{DRIVE_PROCESSED}/anuraset_melspec.h5')
ds_ins   = BioListenDataset(f'{DRIVE_PROCESSED}/insect_melspec.h5')
```

### Cấu trúc thư mục cuối cùng trên Google Drive
```
My Drive/BioListenVN/
├── raw_zips/
│   ├── fsc22-v1.zip                        (~2.5 GB)
│   ├── rfcx-species-audio-detection.zip   (~7.5 GB)
│   ├── anuraset.zip                        (~1.5 GB)
│   ├── insect_Train.zip                    (~4.0 GB)
│   └── insect_Validation.zip               (~1.0 GB)
├── processed/
│   ├── fsc22_melspec.h5
│   ├── rfcx_melspec.h5
│   ├── anuraset_melspec.h5
│   └── insect_melspec.h5
└── metadata/
    ├── fsc22_classes.json
    ├── rfcx_classes.json
    ├── anuraset_classes.json
    ├── anuraset_weak_labels.csv
    ├── insect_classes.json
    └── insect_Annotation.csv
```